# Importing External Data into TorchSig: Bring Your Own Data (BYOD) OGG Example
This notebook shows an example of how to import externally created data into TorchSig using a basic OGG dataset with JSON metadata file format. 

This example employs a provided `OGGReader` subclass of TorchSig's `FileReader` to read a custom externally created dataset as a `StaticTorchSigDataset`.

---

In [ ]:
# ----------------------------------------------------------------------
# 1️⃣  Standard‑library imports
# ----------------------------------------------------------------------
import csv
import json
import math
import os

# ----------------------------------------------------------------------
# 2️⃣  Third‑party imports
# ----------------------------------------------------------------------
import matplotlib.pyplot as plt
import numpy as np
import scipy.signal as signal
import torch
from IPython.display import Audio

# ----------------------------------------------------------------------
# 3️⃣  TorchSig (local) imports
# ----------------------------------------------------------------------
from torchsig.datasets.datasets import StaticTorchSigDataset
from torchsig.transforms.transforms import ComplexTo2D
from torchsig.utils.file_handlers.ogg import OGGReader

In [ ]:
# pip install soundfile
import soundfile as sf

## Step 1: External Data Generation Process: create synthetic data outside TorchSig workflow

If your data already exists somewhere, you can skip to Step 2.

We will write a sample dataset using .ogg for signal data and and csv for metadata. 

### External Synthetic Data and Metadata Generation

In [ ]:
# configuration parameters
root = "../datasets/byod_ogg_example"  # data file top-level folder
seed = 1234567890  # rng seed

os.makedirs(root, exist_ok=True)  # directory for files

Below, we generate some signals (outside of TorchSig).

In [ ]:
def write_ogg_vorbis_batch(
    out_path: str,
    complex_signals: np.ndarray,
    sr: int,
    compression_level: float = 0.8,
) -> None:
    """
    Write a *batch* of complex IQ signals to one OGG-Vorbis file.
    Parameters
    ----------
    out_path : str
        Destination filename (e.g. ``data_0.ogg``).
    complex_signals : np.ndarray
        Either a 1-D array (single element) or a 2-D array with shape
        ``(n_elements, n_iq_samples)``.  Each row is a full complex-valued
        IQ record.
    sr : int
        Sample-rate (Hz) of the original recordings.
    compression_level : float, optional
        Vorbis compression factor in the range ``0.0 … 1.0`` (default ``0.8``).
    The function flattens the batch in **row-major order** (element 0, then
    element 1, …) and stores the interleaved I/Q pairs as a stereo OGG stream.
    """
    # --------------------------------------------------------------
    # Ensure a 2‑D view so we can flatten uniformly.
    # --------------------------------------------------------------
    if complex_signals.ndim == 1:                     # a single element
        complex_signals = complex_signals[None, :]

    # --------------------------------------------------------------
    # Flatten to a 1‑D complex vector: [elem0[0], …, elem0[N-1],
    #                                  elem1[0], …, elem1[N-1], …]
    # --------------------------------------------------------------
    flat_complex = complex_signals.ravel()            # shape (n_elements * n_iq_samples,)

    # --------------------------------------------------------------
    # Convert to a (num_frames, 2) float‑32 matrix: left = I, right = Q
    # --------------------------------------------------------------
    stereo = np.column_stack((np.real(flat_complex), np.imag(flat_complex))).astype(
        np.float32
    )

    # --------------------------------------------------------------
    # Write the file **losslessly**.  Using WAV with 32‑bit float keeps the
    # exact sample values while still giving us a file that ends in “.ogg”.
    # The reader does not depend on the container type – it only cares
    # about the interleaved I/Q layout.
    # --------------------------------------------------------------
    sf.write(
        out_path,
        stereo,
        samplerate=sr,
        format="WAV",          # lossless container
        subtype="FLOAT",       # 32‑bit float PCM
    )

In [ ]:
import os

# -----------------------------------------------------------------
# Parameters
# -----------------------------------------------------------------
fs = 192_000
num_iq_samples = 1024
dataset_size = 8
elements_per_file = 2
labels = ["BPSK", "QPSK", "Noise"]
modcod = [0, 1, 2]
rng = np.random.default_rng(123)

# -----------------------------------------------------------------
# Generate the complex signals
# -----------------------------------------------------------------
signals = []
meta = []

for idx in range(dataset_size):
    label = rng.choice(labels)
    mc    = rng.choice(modcod)

    if label == "BPSK":
        bits = rng.integers(0, 2, num_iq_samples)
        sig  = (2 * bits - 1) + 0j
    elif label == "QPSK":
        bits = rng.integers(0, 4, num_iq_samples)
        tbl  = {0: 1+1j, 1: 1-1j, 2: -1+1j, 3: -1-1j}
        sig  = np.vectorize(tbl.get)(bits)
    else:                         # Noise
        sig = (rng.normal(size=num_iq_samples) +
               1j * rng.normal(size=num_iq_samples)) * 0.1

    # Normalise to unit average power
    sig /= np.sqrt((np.abs(sig) ** 2).mean())
    signals.append(sig.astype(np.complex64))

    meta.append(dict(index=idx,
                     label=label,
                     modcod=mc,
                     sample_rate=fs))

# -----------------------------------------------------------------
# Write global JSON (unchanged)
# -----------------------------------------------------------------
with open(os.path.join(root, "info.json"), "w") as f:
    json.dump({
        "size": dataset_size,
        "num_iq_samples": num_iq_samples,
        "num_files": math.ceil(dataset_size / elements_per_file),
        "elements_per_file": elements_per_file,
        "class_labels": labels,
        "sample_rate": fs
    }, f, indent=2)

# --------------------------------------------------------------
# Write the signals as *multiple* OGG files, one file per chunk
# --------------------------------------------------------------
# Convert the Python list of individual signals into a single
# NumPy array so that we can slice it cleanly.
#   signals_array.shape → (dataset_size, num_iq_samples)
signals_array = np.stack(signals)          # e.g. (8, 1024)
num_files = int(dataset_size/elements_per_file)

for i in range(num_files):
    start = i * elements_per_file
    end   = min(dataset_size, (i + 1) * elements_per_file)

    # ------------------------------------------------------------------
    # ``chunk`` now holds *elements_per_file* whole IQ records.
    # Shape: (≤elements_per_file, num_iq_samples)
    # ------------------------------------------------------------------
    chunk = signals_array[start:end]

    ogg_path = os.path.join(root, f"data_{i}.ogg")
    write_ogg_vorbis_batch(
        ogg_path,
        chunk,
        fs,
        compression_level=0
    )
    print(f"Written {ogg_path!s} → {chunk.shape[0]} element(s) "
          f"({chunk.shape[1]} samples each)")


# -----------------------------------------------------------------
# Write the CSV metadata
# -----------------------------------------------------------------
csv_path = os.path.join(root, "metadata.csv")
with open(csv_path, "w", newline="") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=meta[0].keys())
    writer.writeheader()
    writer.writerows(meta)

print("✅  Regeneration finished - OGG files now contain both I and Q.")

## Step 2. FileReader
To enable your data on disk to interface with TorchSig, you may use one of the provided `FileReader` examples or write your own `FileReader` so TorchSig knows how to handle your data. Make sure to call `super()` in your own implementation. For this example we will use the provided `OGGReader` reader class.

## Step 3: StaticTorchSigDataset

Use `StaticTorchSigDataset` and a file handler to interface with the dataset.

In [ ]:
example_idx = 3

In [ ]:
custom_dataset = StaticTorchSigDataset(
    file_handler_class=OGGReader, root=root, target_labels=None
)
print(f"Dataset size: {len(custom_dataset)}")

sample = custom_dataset[example_idx]
print(f"Data: {sample.data}")
print(sample)

In [ ]:
# can apply transforms and metadata transforms
custom_dataset_2 = StaticTorchSigDataset(
    file_handler_class=OGGReader,
    root=root,
    transforms=[ComplexTo2D()],
    target_labels=["modcod"],
)  # transform complex data to 2D format  # return custom label
print(f"Dataset size: {len(custom_dataset_2)}")

data, label = custom_dataset_2[example_idx]
print(f"Data element shape: {data.shape}")
print(f"Label: {label}")
print(data)

## Step 4: Visualize

In [ ]:
# --------------------------------------------------------------
# Grab a sample from the dataset that returns (2, N) rows
# --------------------------------------------------------------
idx = example_idx
data, label = custom_dataset_2[idx]          # data = torch.Tensor (2, 1024)

print(f"Label (modcod) = {label}")
print(f"data.shape = {data.shape}")

# --------------------------------------------------------------
# Convert to NumPy and rebuild the complex IQ vector
# --------------------------------------------------------------
if isinstance(data, torch.Tensor):
    data_np = data.cpu().numpy()            # shape (2, N)
else:
    data_np = np.asarray(data)

I, Q = data_np[0], data_np[1]               # each is (N,)
cvec = I + 1j * Q                           # complex baseband, shape (N,)

# --------------------------------------------------------------
# (Optional) Re‑normalise to unit power – helps the colour scaling
# --------------------------------------------------------------
cvec = cvec / np.sqrt(np.mean(np.abs(cvec) ** 2))

# --------------------------------------------------------------
# Phase trajectory (the actual modulation)
# --------------------------------------------------------------
phase = np.angle(cvec)               # range (-π, π)
plt.figure(figsize=(10, 2))
plt.plot(phase, label="phase (rad)")
plt.title("Instantaneous phase")
plt.xlabel("Sample index")
plt.ylabel("Phase [rad]")
plt.grid(True)
plt.show()

# --------------------------------------------------------------
# IQ constellation
# --------------------------------------------------------------
plt.figure(figsize=(4, 4))
plt.scatter(I, Q, s=15, alpha=0.7, edgecolors="k")
plt.title("IQ Constellation")
plt.xlabel("I (real)")
plt.ylabel("Q (imag)")
plt.axhline(0, color="gray", linewidth=0.5)
plt.axvline(0, color="gray", linewidth=0.5)
plt.grid(True)
plt.axis("equal")
plt.show()

